# 00 · Data & Methods

Shared foundation for the URL-readability study. Establishes the corpus, the
deduplication rule, and — most importantly — the **segment readability
classifier** that rq1–rq3 depend on. If a number downstream looks wrong, the
cause is almost always a classifier decision documented here.

**Research question (whole project):** *Can we trace a shift from
human-readable to machine-readable URLs on the U.S. federal web, 2004–2024?*

Readability is government-relevant: plain-language / usability guidance
(Plain Writing Act 2010, 21st Century IDEA Act 2018, digital.gov) favors
human-meaningful URLs, and readable URLs are more citable and preservation-
friendly. A measured decline would run counter to that guidance.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root (this notebook lives in analysis/)
sys.path.insert(0, os.path.abspath('.'))
import config, readability as rb, eot_segments as es
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 120); pd.set_option('display.width', 200)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']
CLS_COLORS = {'human':'#2c7fb8','acronym':'#fec44f','machine':'#de2d26'}

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

## 1. Corpus overview — unique URLs per domain per year

Unit of analysis = **one unique URL per crawl year** (dedup on
`(crawl_year, surtkey)`, `dns:` removed). This counts the *URL space a crawl
saw*, not fetch volume. Known gaps: transportation.gov has no 2012; DNS records
exist only in 2024 (excluded).

In [ ]:
rows = []
for dom, path in DBS.items():
    con = duckdb.connect(str(path), read_only=True)
    df = con.sql('''
        SELECT crawl_year, count(*) AS unique_urls FROM (
          SELECT DISTINCT crawl_year, surtkey
          FROM eot_captures WHERE url NOT LIKE 'dns:%'
        ) GROUP BY 1
    ''').df(); con.close()
    df.insert(0,'domain',dom); rows.append(df)
corpus = pd.concat(rows, ignore_index=True)
corpus['crawl_year'] = corpus['crawl_year'].astype(str)
corpus_piv = corpus.pivot(index='domain', columns='crawl_year', values='unique_urls')
corpus_piv = corpus_piv[[y for y in YEAR_ORDER if y in corpus_piv.columns]]
corpus.to_csv('corpus_unique_urls.csv', index=False)
corpus_piv

## 2. The readability classifier

Each path segment is classified `human` / `acronym` / `machine` by
`readability.classify_segment`. Word membership uses **wordfreq** (Zipf
frequency). The active tunables:

In [ ]:
rb.config()

In [ ]:
# Illustrative decisions — this is the table to argue about in review.
demo = ['press-releases','about','national-parks','node','dataset','sites',
        'wp-content','index.cfm','viewpage%3fid','12345','a3f9c1e0b2d4',
        'oia','ibla','blm','foia','uhtbin','id=27','2019-report']
pd.DataFrame({
    'segment': demo,
    'class':   [rb.classify_segment(s) for s in demo],
    'top_token_zipf': [round(max([rb.word_score(t) for t in
        __import__('re').split(r'[^a-z]+', s.lower()) if t] or [0]), 2) for s in demo],
})

### 2a. Validation harness (hand-label a sample)

Classifier quality is the paper's methodological risk, so we quantify it.
This draws a stratified random sample of **distinct** segments, writes it to
`validation_sample.csv` with a blank `true_class` column. Hand-label a few
hundred rows, save, and re-run the agreement cell below to get accuracy and a
confusion matrix. (Re-labeling only needs doing when the classifier changes.)

In [ ]:
SAMPLE_N = 400
seg_all = es.load_segments(DBS)
distinct = seg_all.drop_duplicates(['seg'])[['seg','cls']]
# stratify by predicted class so rare classes are represented
per = max(1, SAMPLE_N // len(rb.CLASSES))
samp = pd.concat([g.sample(min(len(g), per), random_state=0)
                  for _, g in distinct.groupby('cls')], ignore_index=True)
samp = samp.rename(columns={'cls':'pred_class'}); samp['true_class'] = ''
samp[['seg','pred_class','true_class']].to_csv('validation_sample.csv', index=False)
print(f"wrote validation_sample.csv ({len(samp)} rows) — hand-fill true_class, then run next cell")
samp.head(20)

In [ ]:
# Run AFTER hand-labeling validation_sample.csv
try:
    lab = pd.read_csv('validation_sample.csv').dropna(subset=['true_class'])
    lab = lab[lab['true_class'].astype(str).str.strip() != '']
    if len(lab):
        acc = (lab['pred_class'] == lab['true_class']).mean()
        print(f"n labeled = {len(lab)}   accuracy = {acc:.1%}\n")
        print(pd.crosstab(lab['true_class'], lab['pred_class'],
                          rownames=['true'], colnames=['pred']))
    else:
        print("No labels yet — fill in true_class in validation_sample.csv.")
except FileNotFoundError:
    print("Run the sampling cell first.")

## 3. Token-weighted vs type-level (read every RQ figure with both)

- **token-weighted** (weight = URL count `n`): "does a *random page* have a
  readable path?" — reflects the lived URL space, but is dominated by
  high-volume auto-generated sections (e.g. a portal minting millions of
  `/dataset/<id>` URLs).
- **type-level** (each distinct segment counts once): "is the site's *design
  vocabulary* readable?" — immune to volume, but sensitive to rare junk.

The **gap between them** is itself a finding: token↓ while type flat ⇒ machine
content dominated *volume* (portal effect); token↓ and type↓ ⇒ the vocabulary
itself went machine. Helper: `es.readability_pct(df, group_cols, level=...)`.

In [ ]:
# Sanity demo on seg1 (full detail lives in rq1)
demo1 = seg_all[seg_all.pos==1]
print("SEG1 human% — token vs type:")
for lvl in ('token','type'):
    r = es.readability_pct(demo1, 'crawl_year', lvl).set_index('crawl_year')['human']
    print(f"  {lvl:5s}:", r.to_dict())

## 4. The extraction SQL (provenance)

The exact dedup + directory-path + positional-segment SQL every RQ notebook
runs, single-sourced in `eot_segments.py`. Subdomains are excluded by design
(path segments only).

In [ ]:
print(es.segment_sql())